# Thu thập từ API: thời tiết TP.HCM và Hà Nội năm 2023 (Open-Meteo)

Nguồn **API** khác file CSV ở chỗ bạn phải *tự đi lấy*: dựng request, chịu được lỗi mạng và rate
limit (HTTP 429), và **lưu response thô trước khi parse** để parse lại không cần gọi API lần nữa.

API: [Open-Meteo Historical Weather](https://open-meteo.com/en/docs/historical-weather-api), miễn phí,
không cần key, dữ liệu tái phân tích ERA5 (giấy phép CC BY 4.0). Mốc ghim cố định để kết quả tái lập được:

| Thành phố | Toạ độ | Khoảng ngày | Biến theo ngày | Model |
| --- | --- | --- | --- | --- |
| `hcmc` TP.HCM | 10.82, 106.63 | 2023-01-01 → 2023-12-31 | nhiệt độ max/min/mean, tổng mưa | `era5` |
| `hanoi` Hà Nội | 21.03, 105.85 | như trên | như trên | như trên |

Luồng: `build_params` → `fetch_json` (retry) → `collect` (cache `data/raw/open-meteo/*.json`) →
`parse_daily` → `clean` → `insights` → biểu đồ → `write_clean` → `report.md`.

Test **không gọi mạng**: phần gọi API được chấm bằng một `get` giả. Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/basic/01-api-weather-collection -q`

In [ ]:
# explore: tìm gốc repo; dữ liệu thô của bài này do chính bạn thu thập vào data/raw/open-meteo/
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw" / "open-meteo"
CLEAN = ROOT / "data" / "clean" / "open-meteo"
print(RAW)

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

API_URL = "https://archive-api.open-meteo.com/v1/archive"
CITIES = {"hcmc": (10.82, 106.63), "hanoi": (21.03, 105.85)}
START, END = "2023-01-01", "2023-12-31"
DAILY = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean", "precipitation_sum"]
TIMEZONE = "Asia/Bangkok"      # GMT+7, cùng giờ Việt Nam: "một ngày" tính theo giờ địa phương
MODEL = "era5"                 # ghim model: "best_match" có thể đổi nguồn theo thời gian

## 1. Dựng request

`build_params(lat, lon, start, end)` trả dict tham số query: `latitude`, `longitude`, `start_date`,
`end_date`, `daily` (các biến trong `DAILY` nối bằng dấu phẩy), `timezone`, `models`.

In [ ]:
def build_params(lat: float, lon: float, start: str, end: str) -> dict:
    """Tham số query cho API_URL; ghim timezone và model."""
    # TODO
    raise NotImplementedError

## 2. Gọi API chịu lỗi

`fetch_json(get, url, params, max_retries=4, sleep=time.sleep)`:

- gọi `get(url, params=params, timeout=...)` (luôn có timeout; `get` là `requests.get` khi chạy thật, hàm giả khi test);
- `200` → trả `response.json()`;
- **lỗi tạm thời**: status `429` hoặc `5xx`, hoặc `requests.RequestException` (mất mạng, timeout) → chờ rồi thử lại.
  Thời gian chờ lần thứ k (k = 0, 1, 2...) là `2**k` giây (1, 2, 4...), trừ khi response có header
  `Retry-After` thì chờ đúng số giây đó. Chờ bằng hàm `sleep` được truyền vào;
- thử lại tối đa `max_retries` lần (tổng `max_retries + 1` lần gọi), hết lượt thì `raise RuntimeError`;
- lỗi phía client khác (`400`, `404`...) gọi lại cũng vô ích: `raise RuntimeError` ngay, thông báo có mã lỗi.

In [ ]:
def fetch_json(get, url: str, params: dict, max_retries: int = 4, sleep=time.sleep) -> dict:
    """GET có timeout; retry 429/5xx/lỗi mạng với backoff 1, 2, 4... giây (hoặc Retry-After)."""
    # TODO
    raise NotImplementedError

## 3. Thu thập và cache response thô

`collect(get, raw_dir, cities=CITIES, start=START, end=END, sleep=time.sleep) -> list[Path]`:
với mỗi thành phố, tệp cache là `raw_dir / f"{city}_{start}_{end}.json"`.

- đã có tệp → **không gọi API** (chạy lại notebook không tốn request, không phụ thuộc mạng);
- chưa có → `fetch_json(...)`, ghi **nguyên response** ra tệp tạm trong `raw_dir` rồi `replace` sang tên thật
  (không bao giờ để lại tệp JSON dở nếu bị ngắt giữa chừng);
- trả danh sách đường dẫn theo thứ tự `cities`.

In [ ]:
def collect(get, raw_dir: Path, cities: dict = CITIES, start: str = START, end: str = END,
            sleep=time.sleep) -> list[Path]:
    """Mỗi thành phố một tệp raw_dir/<city>_<start>_<end>.json; có rồi thì không gọi API."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: thu thập (gọi API thật, mỗi thành phố một request; lần sau đọc cache)
paths = collect(requests.get, RAW)
for p in paths:
    print(p.name, f"{p.stat().st_size:,} byte")

## 4. Parse và làm sạch

- `parse_daily(payload, city)`: từ khoá `daily` của response, DataFrame cột
  `city, date, tmax, tmin, tmean, precip_mm` (`date` kiểu datetime).
- `clean(df)`: bỏ ngày mà cả 4 số đo đều null (API trả `null` cho ngày chưa có dữ liệu), bỏ trùng
  `(city, date)` giữ bản lấy sau cùng (hai lần thu thập chồng khoảng ngày), sắp theo `city, date`, đánh lại index.

In [ ]:
def parse_daily(payload: dict, city: str) -> pd.DataFrame:
    """Cột city, date, tmax, tmin, tmean, precip_mm."""
    # TODO
    raise NotImplementedError


def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Bỏ ngày toàn null, bỏ trùng (city, date) giữ bản sau, sắp theo city, date."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: từ JSON thô tới bảng sạch
df = clean(pd.concat([parse_daily(json.loads(p.read_text(encoding="utf-8")), p.name.split("_")[0]) for p in paths]))
print(df.groupby("city").size())
df.describe()

## 5. Insight

`insights(df)` trả `dict` **theo thành phố**: `{"hcmc": {...}, "hanoi": {...}}`, mỗi thành phố các khoá:

| Khoá | Nghĩa |
| --- | --- |
| `annual_rain_mm` | tổng lượng mưa |
| `rainy_season_share` | tỉ lệ lượng mưa rơi trong tháng 5–10 |
| `hottest_month`, `coldest_month` | tháng (1–12) có `tmean` trung bình cao nhất / thấp nhất |
| `amplitude_c` | biên độ nhiệt năm: trung bình `tmean` của tháng nóng nhất trừ tháng lạnh nhất |
| `max_temp`, `max_temp_date` | `tmax` cao nhất và ngày đó (`"YYYY-MM-DD"`) |
| `hot_days` | số ngày `tmax` ≥ 35 °C (ngưỡng nắng nóng của Trung tâm Dự báo KTTV) |
| `rainy_days` | số ngày mưa ≥ 1 mm |

In [ ]:
def insights(df: pd.DataFrame) -> dict:
    """{city: {annual_rain_mm, rainy_season_share, hottest_month, coldest_month, amplitude_c,
              max_temp, max_temp_date, hot_days, rainy_days}}"""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: các con số để viết report.md
INSIGHTS = insights(df)
pd.DataFrame(INSIGHTS)

## 6. Biểu đồ

Ít nhất hai biểu đồ (mỗi cái một cell `explore`):

1. nhiệt độ trung bình theo tháng của hai thành phố trên cùng trục: biên độ khác nhau thế nào?
2. lượng mưa theo tháng của hai thành phố (cột cạnh nhau): mùa mưa bắt đầu, kết thúc khi nào?

In [ ]:
# explore: nhiệt độ trung bình theo tháng
# TODO: hai đường tmean trung bình theo tháng, một đường mỗi thành phố

In [ ]:
# explore: lượng mưa theo tháng
# TODO: cột lượng mưa theo tháng, hai thành phố cạnh nhau

## 7. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi đè `out_dir/daily.parquet` (qua tệp tạm) và trả đường dẫn.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/daily.parquet (qua tệp tạm), trả đường dẫn."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: ghi data/clean/open-meteo/daily.parquet
path = write_clean(df, CLEAN)
print(path, len(pd.read_parquet(path)), "dòng")

## 8. Báo cáo

Viết `report.md` cạnh notebook: `## Bộ dữ liệu` (API, tham số đã ghim, giấy phép), `## Làm sạch`,
`## Insight` (có số), `## Giới hạn` (ERA5 là tái phân tích lưới ~25 km, không phải trạm đo).